# E3–E4: LoRA Target-Module and Layer Placement Study — CheXpert

Study which attention projections and which transformer layers benefit most from LoRA adaptation.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Inspect Model Architecture

Examine DINOv2 ViT-B/14 transformer blocks to identify attention projections available for LoRA adaptation.

In [ ]:
"""
Inspect DINOv2 ViT-B/14 transformer layers and identify target attention projection modules.
"""
from src.models import get_attention_layer_names, load_backbone

backbone, _ = load_backbone()
layer_names: list[str] = get_attention_layer_names(backbone)
print(f"Found {len(layer_names)} attention-related modules")
for name in layer_names[:10]:
    print(f"  {name}")

## 2. E3a — Q+V Only (baseline)

Apply LoRA (rank 16) only to Query and Value attention projections across all 12 transformer layers.

In [ ]:
"""
Train and evaluate E3a baseline: LoRA injected strictly into Query and Value projections across all layers.
"""
from typing import Any
import torch
import torch.nn as nn
from torch.optim import AdamW
from src.data import build_chexpert_dataloaders, build_transforms
from src.evaluation import Evaluator
from src.models import apply_lora, count_trainable_parameters, load_backbone, MultiLabelHead
from src.training.losses import MultiLabelBCELoss
from src.utils.config import load_config
from src.utils.logging_utils import ExperimentLogger

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
train_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=True)
val_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=False)
train_loader, val_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
evaluator = Evaluator(task="multilabel", label_names=cfg["dataset"]["label_names"])
logger = ExperimentLogger()
study_records: list[dict[str, Any]] = []

def train_and_eval_variant(
    variant_id: str,
    target_modules: list[str],
    layers_pattern: str | None = None,
    rank: int = 16,
) -> dict[str, Any]:
    """
    Builds, trains for 8 epochs, and evaluates a specific LoRA placement configuration.
    """
    bb, _ = load_backbone()
    lora_params: dict[str, Any] = {"r": rank, "lora_alpha": 16, "target_modules": target_modules}
    if layers_pattern is not None:
        lora_params["layers_pattern"] = layers_pattern
    bb = apply_lora(bb, lora_params)
    hd = MultiLabelHead(in_features=768, num_labels=14)

    class LayerStudyModel(nn.Module):
        def __init__(self, encoder: nn.Module, classifier: nn.Module) -> None:
            super().__init__()
            self.encoder = encoder
            self.classifier = classifier
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            out = self.encoder(pixel_values=x)
            return self.classifier(out.last_hidden_state[:, 0, :])

    m = LayerStudyModel(bb, hd).to(device)
    tr, tot = count_trainable_parameters(m)
    crit = MultiLabelBCELoss()
    opt = AdamW(
        [
            {"params": [p for p in m.encoder.parameters() if p.requires_grad], "lr": 2e-4},
            {"params": m.classifier.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    for epoch in range(8):
        m.train()
        for batch in train_loader:
            pv = batch["pixel_values"].to(device)
            lb = batch["labels"].to(device)
            opt.zero_grad()
            loss = crit(m(pv), lb)
            loss.backward()
            opt.step()

    eval_res = evaluator.evaluate(m, val_loader, device=str(device))
    auc: float = float(eval_res["mean_auroc"])
    rec = {
        "variant": variant_id,
        "target_modules": ",".join(target_modules),
        "layers_pattern": layers_pattern if layers_pattern else "all (0-11)",
        "rank": rank,
        "trainable_params": tr,
        "val_auroc": auc,
    }
    logger.log(
        run_id=f"study_{variant_id}",
        dataset="CheXpert",
        backbone="DINOv2 ViT-B/14",
        method=variant_id,
        rank=rank,
        trainable_params=tr,
        val_auroc=auc,
    )
    return rec

res_e3a = train_and_eval_variant("E3a_QV_Only", ["query", "value"])
study_records.append(res_e3a)
print("E3a Outcome:", res_e3a)

## 3. E3b — All Attention Projections

Expand target modules to include all four projection matrices: `query`, `key`, `value`, and `output`.

In [ ]:
"""
Train and evaluate E3b configuration: LoRA applied to Query, Key, Value, and Output projections.
"""
res_e3b = train_and_eval_variant("E3b_All_Attention", ["query", "key", "value", "output"])
study_records.append(res_e3b)
print("E3b Outcome:", res_e3b)

## 4. E3c — Upper-Quarter Layers Only

Apply LoRA adapters exclusively to the final quarter of transformer blocks (blocks 9, 10, 11) using `layer_selection.enabled=True`, `strategy=upper_quarter`.

In [ ]:
"""
Train and evaluate E3c configuration: LoRA restricted to upper-quarter transformer blocks (blocks 9, 10, 11).
"""
res_e3c = train_and_eval_variant("E3c_Upper_Quarter", ["query", "value"], layers_pattern=r"layer\.(9|10|11)\.")
study_records.append(res_e3c)
print("E3c Outcome:", res_e3c)

## 5. E3d — Lower Layers vs Distributed

Compare adapting lower blocks (blocks 0, 1, 2) against distributing adaptation across alternate blocks (blocks 0, 2, 4, 6, 8, 10).

In [ ]:
"""
Train and evaluate E3d comparative variants: lower-layer adaptation versus distributed interleaved adaptation.
"""
res_e3d_lower = train_and_eval_variant("E3d_Lower_Layers", ["query", "value"], layers_pattern=r"layer\.(0|1|2)\.")
study_records.append(res_e3d_lower)
print("E3d Lower Layers Outcome:", res_e3d_lower)

res_e3d_dist = train_and_eval_variant("E3d_Distributed", ["query", "value"], layers_pattern=r"layer\.(0|2|4|6|8|10)\.")
study_records.append(res_e3d_dist)
print("E3d Distributed Outcome:", res_e3d_dist)

## 6. E4 — Layer-Selective LoRA (candidate contribution)

Evaluate candidate contribution: concentrate adaptation capacity into the top-performing layers with parameter matching (e.g., r=32 on the upper 4 blocks 8-11).

In [ ]:
"""
Train and evaluate E4 candidate contribution: layer-selective LoRA matched in total parameters against uniform baseline.
"""
res_e4 = train_and_eval_variant("E4_Layer_Selective_r32", ["query", "value"], layers_pattern=r"layer\.(8|9|10|11)\.", rank=32)
study_records.append(res_e4)
print("E4 Layer-Selective Outcome:", res_e4)

## 7. Results Comparison Table and Plots

Compile all target-module and layer-placement study results into a consolidated comparison table and dual-axis chart.

In [ ]:
"""
Tabulate comparative results and render dual-axis visualization of validation AUROC and trainable parameters.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

study_df = pd.DataFrame(study_records)
print(study_df[["variant", "target_modules", "layers_pattern", "rank", "trainable_params", "val_auroc"]].to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(12, 6))
x_indices = range(len(study_df))

ax1.bar(x_indices, study_df["val_auroc"], width=0.4, color="#2b5c8f", align="center", label="Validation AUROC")
ax1.set_ylabel("Validation AUROC", color="#2b5c8f", fontsize=12)
ax1.set_ylim(0.75, 0.88)
ax1.set_xticks(list(x_indices))
ax1.set_xticklabels(study_df["variant"], rotation=30, ha="right", fontsize=11)

ax2 = ax1.twinx()
ax2.plot(x_indices, study_df["trainable_params"], color="#d95f02", marker="s", linewidth=2.5, label="Trainable Parameters")
ax2.set_ylabel("Trainable Parameters", color="#d95f02", fontsize=12)
ax2.grid(False)

plt.title("E3–E4: LoRA Module Targets and Layer Placement Comparison", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()